# 05_continuous_batching_and_scheduling: Continuous Batching Simulation

This notebook implements a queue scheduler simulation comparing Static Batching vs. Continuous Batching (Iteration-Level Scheduling). We log active batch size and queue latency metrics to verify serving throughput optimizations.

### Scheduling Logic
- **Static Batching**: Groups $B$ sequences, padding all entries to matching lengths, running till completion before accepting the next batch.
- **Continuous Batching**: Steps sequence execution iteratively. Completed sequences are immediately evicted, and queued sequences are inserted at active token cycles.

In [1]:
import os
import random
import copy

class ServingRequest:
    def __init__(self, req_id, arrival_time, prompt_len, decode_len):
        self.req_id = req_id
        self.arrival_time = arrival_time
        self.prompt_len = prompt_len
        self.decode_len = decode_len
        self.steps_completed = 0
        self.ttft = None
        self.finish_time = None

def simulate_static_batching(requests, batch_size=4):
    time = 0
    completed = []
    queue = sorted(requests, key=lambda r: r.arrival_time)
    
    while queue:
        batch = queue[:batch_size]
        queue = queue[batch_size:]
        
        max_prompt = max(r.prompt_len for r in batch)
        max_decode = max(r.decode_len for r in batch)
        
        time += max_prompt
        for r in batch:
            r.ttft = time - r.arrival_time
            
        time += max_decode
        for r in batch:
            r.finish_time = time
            completed.append(r)
            
    return completed

In [2]:
def simulate_continuous_batching(requests, max_concurrency=4):
    time = 0
    queue = sorted(requests, key=lambda r: r.arrival_time)
    active = []
    completed = []
    
    while queue or active:
        while len(active) < max_concurrency and queue and queue[0].arrival_time <= time:
            req = queue.pop(0)
            active.append(req)
            
        if not active:
            time = queue[0].arrival_time
            continue
            
        for req in list(active):
            if req.steps_completed == 0:
                req.ttft = time - req.arrival_time
            
            req.steps_completed += 1
            
            if req.steps_completed >= (req.prompt_len + req.decode_len):
                req.finish_time = time
                completed.append(req)
                active.remove(req)
                
        time += 1
        
    return completed

In [3]:
# Test the simulator with 8 requests
random.seed(42)
test_requests = [
    ServingRequest(f"R_{i}", arrival_time=i * 2, prompt_len=random.randint(5, 15), decode_len=random.randint(10, 30))
    for i in range(8)
]

# Run separate copies to avoid object reference conflicts
static_res = simulate_static_batching(copy.deepcopy(test_requests), batch_size=4)
cont_res = simulate_continuous_batching(copy.deepcopy(test_requests), max_concurrency=4)

avg_ttft_static = sum(r.ttft for r in static_res) / len(static_res)
avg_tpot_static = sum(r.finish_time - r.arrival_time for r in static_res) / len(static_res)

avg_ttft_cont = sum(r.ttft for r in cont_res) / len(cont_res)
avg_tpot_cont = sum(r.finish_time - r.arrival_time for r in cont_res) / len(cont_res)

print(f"Static Batching - Average TTFT:     {avg_ttft_static:.2f} cycles")
print(f"Static Batching - Average Latency:  {avg_tpot_static:.2f} cycles")
print(f"Continuous Batching - Average TTFT:    {avg_ttft_cont:.2f} cycles")
print(f"Continuous Batching - Average Latency: {avg_tpot_cont:.2f} cycles")

Static Batching - Average TTFT:     24.50 cycles
Static Batching - Average Latency:  47.50 cycles
Continuous Batching - Average TTFT:    8.00 cycles
Continuous Batching - Average Latency: 33.38 cycles


### Output Explanation & Verification

- **TTFT Overhead**: Under Static Batching, requests wait for the active batch to complete before processing, elevating average TTFT to **24.50 cycles** (vs. **8.00 cycles** for Continuous Batching).
- **Continuous Scheduling Latency**: Continuous Batching dynamically routes sequences, dropping average request completion latency to **33.38 cycles** (vs. **47.50 cycles** for Static Batching). This validates why iteration-level queue scheduling is preferred in production serving environments.